Let's illustrate this with a practical code example. We will build a simple Retrieval-Augmented Generation (RAG) application with multiple stages:

Query Generation: Generate a suitable query based on the user's question to retrieve relevant context.
Context Retrieval: Fetch context using the generated query.
Answer Generation: Produce a final answer based on the retrieved context and the original question.
The code implementation for this multi-stage program is shown below.

In [1]:
import os
os.environ['HTTP_PROXY'] = 'http://127.0.0.1:7890'
os.environ['HTTPS_PROXY'] = 'http://127.0.0.1:7890'

In [ ]:
import dspy

class QueryGenerator(dspy.Signature):
    """Generate a query based on question to fetch relevant context"""
    question: str = dspy.InputField()
    query: str = dspy.OutputField()

def search_wikipedia(query: str) -> list[str]:
    """Query ColBERT endpoint, which is a knowledge source based on wikipedia data"""
    try:
        results = dspy.ColBERTv2(url='http://20.102.90.50:2017/wiki17_abstracts')(query, k=1)
    except Exception as e:
        print(f"Error occurred while searching Wikipedia: {e}")
        return []
    return [x["text"] for x in results]

class RAG(dspy.Module):
    def __init__(self):
        self.query_generator = dspy.Predict(QueryGenerator)
        self.answer_generator = dspy.ChainOfThought("question,context->answer")

    def forward(self, question, **kwargs):
        query = self.query_generator(question=question).query
        context = search_wikipedia(query)[0]
        return self.answer_generator(question=question, context=context).answer

In [4]:
import dspy

import os
import json

cur_path = os.getcwd()
model_conf_path = os.path.join(cur_path, "../api_key")

model_config_dict = json.load(open(os.path.join(model_conf_path, "mymodelkey.json")))

used_model = "qwen3-8b" # "qwen3-8b" # "llama3.1"

your_openai_api_key = model_config_dict[used_model]["api_key"]
your_openai_base_url = model_config_dict[used_model]["base_url"]
your_openai_compatible_model = model_config_dict[used_model]["model"]
your_openai_other_kwargs = model_config_dict[used_model].get("other_kwargs", {})

os.environ["OPENAI_API_KEY"] = f"{your_openai_api_key}"
os.environ["OPENAI_API_BASE"] = f"{your_openai_base_url}"

dspy.settings.configure(lm=dspy.LM(your_openai_compatible_model, **your_openai_other_kwargs))


In [6]:
rag = RAG()
print(rag(question="Is Lebron James the basketball GOAT?"))

Error occurred while searching Wikipedia: 'topk'


IndexError: list index out of range